# Charge cycles for SoH — two ideas tested, both rejected

**Question:** can charge cycles give us (1) a *range-fade health signal*, or (2) a useful *model feature*?

**Answer: no on both** — and Part 2 is a cautionary tale about a correlation that lies.

Data: **Bajaj** (the one OEM with a real BMS cycle counter + odometer + Wh/km efficiency), plus Euler/Mahindra for the equiv-cycle proxy. Target = reported SoH. See `docs/OEM_MODELING_PLAYBOOK.md` §7 insight 8.

In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor
bj = pd.read_parquet('data/bajaj/features/feature_table.parquet').sort_values(['vin','age_months'])
print('Bajaj feature table:', bj.shape, '|', bj.vin.nunique(), 'vehicles')

## Part 2 — range-fade health signal

`km/cycle` = capacity x DoD-per-cycle x efficiency(km/Wh). To isolate capacity we divide out efficiency:
`Wh/cycle = (km/cycle) x Wh/km`. Then check how well each tracks reported SoH **within each vehicle**.

In [ ]:
def metrics(g):
    g = g[(g['km_month']>0)&(g['cyc_month']>0)&g['driveeff_mean'].notna()].copy()
    g['kmpc'] = g['km_month']/g['cyc_month']
    g['whpc'] = g['kmpc']*g['driveeff_mean']
    for c in ['kmpc','whpc']:
        g[c] = g[c].clip(g[c].quantile(.05), g[c].quantile(.95)).rolling(3,min_periods=1,center=True).median()
    return g

def naive_rho(col):
    rs=[]
    for v,g in bj.groupby('vin'):
        g=metrics(g)
        if len(g)>=5 and g['soh'].nunique()>1: rs.append(spearmanr(g[col], g['soh']).statistic)
    rs=np.array([r for r in rs if np.isfinite(r)])
    return np.median(rs), (rs>0.3).mean(), len(rs)

for col,lab in [('kmpc','km/cycle (raw)'),('whpc','Wh/cycle (eff. removed)')]:
    m,f,n = naive_rho(col)
    print(f'{lab:24s}: naive within-vehicle rho vs SoH = {m:+.2f}  ({f*100:.0f}% of {n} track)')

Wh/cycle scores **+0.88** — looks like a strong health signal. Two example vehicles:

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.8))
for ax,v in zip(axes, ['MD2B94305RWG59975','MD2B94305RWG63864']):
    g=metrics(bj[bj.vin==v]); base=g['whpc'].iloc[:3].median(); ax2=ax.twinx()
    ax.plot(g['age_months'], g['soh'], 'o-', color='tab:blue')
    ax2.plot(g['age_months'], 100*g['whpc']/base, 's--', color='tab:orange')
    ax.set_title(v[-8:]); ax.set_xlabel('age (months)')
    ax.set_ylabel('reported SoH %', color='tab:blue'); ax2.set_ylabel('Wh/cycle retention %', color='tab:orange')
fig.suptitle('Looks convincing — both decline together. But is it real?'); fig.tight_layout()

### The re-check: remove the shared age trend

Both SoH and Wh/cycle decline monotonically over the same ~10-month window, so a within-vehicle correlation is inflated by that **common time trend**. The honest tests: correlate the **age-residuals**, and the **month-to-month changes**.

In [ ]:
def resid(y,x):
    x=np.asarray(x,float); y=np.asarray(y,float); b=np.polyfit(x,y,1); return y-(b[0]*x+b[1])

def recheck(col):
    raw,detr,diff=[],[],[]
    for v,g in bj.groupby('vin'):
        g=metrics(g)
        if len(g)<6 or g['soh'].nunique()<3: continue
        soh=g['soh'].to_numpy(); age=g['age_months'].to_numpy(); c=g[col].to_numpy()
        raw.append(spearmanr(c,soh).statistic)
        detr.append(spearmanr(resid(c,age), resid(soh,age)).statistic)
        dc,ds=np.diff(c),np.diff(soh)
        if np.std(dc)>0 and np.std(ds)>0: diff.append(spearmanr(dc,ds).statistic)
    return np.nanmedian(raw), np.nanmedian(detr), np.nanmedian(diff)

print(f"{'metric':24s}{'naive':>10s}{'age-controlled':>16s}{'month-to-month':>16s}")
for col,lab in [('kmpc','km/cycle (raw)'),('whpc','Wh/cycle')]:
    r0,r1,r2=recheck(col); print(f'{lab:24s}{r0:>+10.2f}{r1:>+16.2f}{r2:>+16.2f}')

**Verdict (Part 2):** the +0.88 was an **artifact of the shared age trend**. Age-controlled, Wh/cycle drops to ~0, and the efficiency decomposition did *worse* than raw km/cycle. Energy-per-cycle is **not** a usable health signal.

## Part 1 — cycles as a model feature

Leave-one-vehicle-out RMSE predicting reported SoH, averaged over **8 fixed seeds** (so the ordering isn't noise).

In [ ]:
bj['inv_sqrt_age']=1/np.sqrt(bj['age_months']+1)
sets={'age only':['age_months','inv_sqrt_age'],
      'age + cum_km':['age_months','inv_sqrt_age','cum_km'],
      'age + cycles':['age_months','inv_sqrt_age','cum_cycles','cyc_month'],
      'age + cycles + thermal':['age_months','inv_sqrt_age','cum_cycles','cyc_month','temp_mean','temp_p95']}
def lovo(feats,seed):
    m=bj.dropna(subset=feats+['soh']); errs=[]
    for v in m.vin.unique():
        tr,te=m[m.vin!=v],m[m.vin==v]
        if len(te)<3 or len(tr)<30: continue
        mdl=GradientBoostingRegressor(n_estimators=200,max_depth=2,learning_rate=0.05,subsample=0.8,random_state=seed)
        mdl.fit(tr[feats],tr['soh']); errs.append(np.sqrt(np.mean((te['soh']-mdl.predict(te[feats]))**2)))
    return np.mean(errs)
for name,fs in sets.items():
    vals=[lovo(fs,s) for s in range(8)]
    print(f'{name:24s}: {np.mean(vals):.2f} +/- {np.std(vals):.2f} pp')

In [ ]:
# equiv-cycle proxy (Ah throughput) for Euler/Mahindra: does cycle-rate predict monthly SoH loss?
def loss_corr(m, col, nominal):
    m=m.sort_values(['vin','age_months']).copy()
    m['loss']=m.groupby('vin')['soh'].diff(-1).clip(lower=0)
    j=pd.DataFrame({'loss':m['loss'],'rate':m[col]/(2*nominal)}).dropna()
    return spearmanr(j['rate'],j['loss']).statistic, len(j)
for oem,nom in [('euler',133.0),('mahindra',100.0)]:
    m=pd.read_parquet(f'data/{oem}/features/feature_table.parquet')
    r,n=loss_corr(m,'ah_throughput',nom)
    print(f'{oem:8s}: equiv-cycle rate vs monthly SoH loss  rho={r:+.2f} (n={n})')

**Verdict (Part 1):** adding cycle features *worsens* the model (age-only is best; std ~0.02, so the ordering is real), and the equiv-cycle proxy is null for Euler/Mahindra. Calendar age already captures the aging; cycles are collinear -> overfit.

---

### Methodological rule
Never trust a *'tracks SoH'* claim from a within-vehicle correlation over a short, monotone window — **age-control it first** (residuals or month-to-month deltas). Use cycles only for warranty-km projection / cycle-RUL (see `../04_forecast_decisions/bajaj_warranty_km_cycle_rul.ipynb`), not for health.